# Chapter 12: Containers, Service Containers and GHCR (notebook edition)

## Learning Objectives

- Build GHCR references correctly
- Verify a published image by digest
- Explain the two classic push failures

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Image references

GHCR needs a lowercase repository. We build references and check the rule that failed in the real run.

In [ ]:
from intro_gha.images import ghcr_ref, repository_name_ok
print(repository_name_ok("ghcr.io/Friend09/tally:test"), repository_name_ok(ghcr_ref("Friend09", "tally", tag="latest")))
assert not repository_name_ok("ghcr.io/Friend09/tally:test")
assert ghcr_ref("Friend09", "tally", tag="latest") == "ghcr.io/friend09/tally:latest"

## 2. Digest verification

The `publish` job pushed an image and exported its digest; `verify` pulled that digest and ran it. Both reports should carry the same digest, and the run should print 5.

In [ ]:
import json
run = json.loads((ROOT/"fixtures"/"containers_ch12_run.json").read_text())["reports"]
d = run["publish"]["digest"]
print(d[:19], run["verify"]["verify_output"])
assert d == run["verify"]["digest"] and run["verify"]["verify_output"] == "5"

## 3. The two failed pushes

Uppercase owner, and a push without `packages: write`.

In [ ]:
s = json.loads((ROOT/"fixtures"/"containers_ch12_summary.json").read_text())
print(s["uppercase"]["error"][:80]); print(s["denied"]["error"])
assert run["uppercase_tag"]["uppercase_build_outcome"] == "failure" and run["publish_denied"]["denied_push_outcome"] == "failure"

## 4. Container job and service container

A git-less Debian container still checked out the repo; Redis answered PING.

In [ ]:
ic = run["in_container"]
print(ic["container_os"], "| git:", ic["git"], "| pyproject:", ic["has_pyproject"], "| redis:", run["service"]["redis_reply"])
assert ic["git"] == "none" and ic["has_pyproject"] == "yes" and run["service"]["redis_reply"] == "+PONG"

## 5. Image size arithmetic

Compressed bytes in MiB, and what 20 pulls move.

In [ ]:
b = s["publish"]["compressed_bytes"]
print(round(b / 1_048_576, 2), "MiB;", 20 * b, "bytes for 20 pulls")
assert round(b / 1_048_576, 2) == 45.05 and 20 * b == 944_720_760

## Takeaways & Next Steps

- Lowercase the owner; give only the pushing job `packages: write`.
- Deploy by digest, not tag.
- Next: Chapter 13, continuous deployment and environments.

## Chapter Link

Read: `learning_modules/chapter_12_containers_and_ghcr.md`